In [6]:
# INSTALL
!pip install -q sentence-transformers faiss-cpu pypdf numpy

# IMPORTS
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json
import os

# UPLOAD PDF
from google.colab import files
uploaded = files.upload()  # upload training_document.pdf when prompted

# STEP 1 - Extract text
print("Extracting text...")
reader = PdfReader("training_document.pdf")
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"
print(f"Total characters: {len(full_text)}")

# STEP 2 - Chunk
print("Chunking...")
def chunk_text(text, chunk_size=1000, chunk_overlap=90):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        if end >= len(text):
            chunks.append(text[start:].strip())
            break
        chunk = text[start:end]
        last_period = chunk.rfind('.')
        if last_period > chunk_size // 2:
            end = start + last_period + 1
        chunks.append(text[start:end].strip())
        next_start = end - chunk_overlap
        start = next_start if next_start > start else start + chunk_size // 2
    chunks = [c for c in chunks if len(c) > 50]
    return chunks

chunks = chunk_text(full_text)
print(f"Total chunks: {len(chunks)}")

# STEP 3 - Save chunks
with open("chunks.json", "w") as f:
    json.dump(chunks, f)
print("Chunks saved")

# STEP 4 - Generate embeddings
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Generating embeddings...")
embeddings = model.encode(chunks, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

# STEP 5 - Save FAISS index
print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
faiss.write_index(index, "faiss_index.index")
print(f"FAISS index saved. Total vectors: {index.ntotal}")

print("\nDONE! Downloading files...")

# DOWNLOAD both files automatically
files.download("chunks.json")
files.download("faiss_index.index")

Extracting text...
Total characters: 68253
Chunking...
Total chunks: 89
Chunks saved
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Embeddings shape: (89, 384)
Building FAISS index...
FAISS index saved. Total vectors: 89

DONE! Downloading files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>